# Clean tables and rasters which already have RUE and CV and save one large table

In [ ]:
"""
Clean yearly tables + rasters consistently, using a shared pixel ID.

Cleaning rules (pixel-level, applied across ALL years at once):
  1. Drop any pixel where precipitation > PRECIP_THRESHOLD in ANY year.
  2. Drop any pixel with an NA in ANY non-exempt variable, in ANY year.

A pixel that survives both checks gets a stable integer `location_id`. That same
ID is used both in the cleaned tables and as a dedicated ID band/raster, so
you can always re-link table rows <-> raster pixels later.

Outputs:
  - location_id_lookup.parquet            (lon_r, lat_r, location_id) for every kept pixel
  - tables/table_wgs84_<year>_clean.parquet   (per year)
  - tables/table_wgs84_2002-2023_clean.parquet (all years, one file)
  - tables/table_wgs84_example.parquet         (small example subset)
  - rasters/location_id.tif                       (single-band ID raster, same for every year)
  - rasters/stack_wgs84_<year>_clean.tif       (per year, all original bands, cleaned)
"""

In [1]:
import numpy as np
import pandas as pd
import rasterio
from pathlib import Path

In [11]:
# ======================================================================
# ---------- Config - adjust these to match your setup ----------
# ======================================================================

years = list(range(2002, 2024))  # 2002-2023 inclusive

# --- input paths ---
parquet_paths = {
    y: f"/Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/table_wgs84_{y}_rue_cv.parquet"
    for y in years
}
tif_paths = {
    y: f"/Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_{y}_rue.tif"
    for y in years
}

# --- output paths ---
out_dir_tables = Path("/Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/")
out_dir_rasters = Path("/Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/")

COORD_DECIMALS = 6

# Precipitation column name and threshold for filtering pixels
PRECIP_VAR = "pr_sum"
PRECIP_THRESHOLD = 1000.0  # mm

# Variables not allowed to have NAs
NA_VARS = ["Npp", "tmmx_mean", "tmmn_mean", "pr_sum", "veg_tmmx_mean", "veg_tmmn_mean", "veg_pr_sum", "vegetation_length", "elevation_mean", "longitude", "latitude", "year", "rue", "pr_sum_cv", "veg_pr_sum_cv", "Npp_cv"]

# TODO: how many pixels (not rows) in the small example table
N_EXAMPLE_PIXELS = 100000

ID_COL = "location_id"
RANDOM_SEED = 42

In [3]:
# ======================================================================
# ---------- Step 1: load all yearly tables (all variables) ----------
# ======================================================================

dfs = []
for y, p in parquet_paths.items():
    d = pd.read_parquet(p)  # all columns
    d["lon_r"] = d["longitude"].round(COORD_DECIMALS)
    d["lat_r"] = d["latitude"].round(COORD_DECIMALS)
    # "year" column is assumed to already exist in the data
    dfs.append(d)

long_df = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(long_df)} rows across {len(years)} years")

na_check_vars = NA_VARS
print(f"Checking NAs on {len(na_check_vars)} variables: {na_check_vars}")

Loaded 92801060 rows across 22 years
Checking NAs on 16 variables: ['Npp', 'tmmx_mean', 'tmmn_mean', 'pr_sum', 'veg_tmmx_mean', 'veg_tmmn_mean', 'veg_pr_sum', 'vegetation_length', 'elevation_mean', 'longitude', 'latitude', 'year', 'rue', 'pr_sum_cv', 'veg_pr_sum_cv', 'Npp_cv']


In [4]:
# ======================================================================
# ---------- Step 2: determine which pixels to exclude ----------
# ======================================================================

# a) precipitation exceeds threshold in ANY year
precip_bad = (
    long_df.loc[long_df[PRECIP_VAR] > PRECIP_THRESHOLD, ["lon_r", "lat_r"]]
    .drop_duplicates()
)
print(f"{len(precip_bad)} pixels exceed {PRECIP_THRESHOLD} ({PRECIP_VAR}) in >=1 year")

# b) NA in any non-exempt variable in ANY year
na_mask = long_df[na_check_vars].isna().any(axis=1)
na_bad = long_df.loc[na_mask, ["lon_r", "lat_r"]].drop_duplicates()
print(f"{len(na_bad)} pixels have an NA in a non-exempt variable in >=1 year")

bad_pixels = pd.concat([precip_bad, na_bad], ignore_index=True).drop_duplicates()
print(f"{len(bad_pixels)} pixels excluded in total")

all_pixels = long_df[["lon_r", "lat_r"]].drop_duplicates()
good_pixels = (
    all_pixels.merge(bad_pixels, on=["lon_r", "lat_r"], how="left", indicator=True)
    .query('_merge == "left_only"')
    .drop(columns="_merge")
    .reset_index(drop=True)
)
print(f"{len(good_pixels)} / {len(all_pixels)} pixels kept")

1197011 pixels exceed 1000.0 (pr_sum) in >=1 year
631123 pixels have an NA in a non-exempt variable in >=1 year
1795136 pixels excluded in total
2423094 / 4218230 pixels kept


In [5]:
# ======================================================================
# ---------- Step 3: assign a stable pixel/location ID ----------
# ======================================================================

# sort so IDs are reproducible across reruns (order itself is arbitrary)
good_pixels = good_pixels.sort_values(["lat_r", "lon_r"], ascending=[False, True]).reset_index(drop=True)
good_pixels[ID_COL] = np.arange(len(good_pixels), dtype=np.int64)

id_lookup_path = out_dir_tables / "location_id_lookup.parquet"
good_pixels.to_parquet(id_lookup_path, index=False)
print(f"Saved location ID lookup ({len(good_pixels)} pixels) to {id_lookup_path}")

Saved location ID lookup (2423094 pixels) to /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/location_id_lookup.parquet


In [6]:
# ======================================================================
# ---------- Step 4: filter table to good pixels, attach ID ----------
# ======================================================================

clean_df = long_df.merge(good_pixels, on=["lon_r", "lat_r"], how="inner")
print(f"Clean table: {len(clean_df)} rows, {clean_df[ID_COL].nunique()} pixels")

# sanity check: every kept pixel should still have all years
counts = clean_df.groupby(ID_COL)["year"].nunique()
n_incomplete = (counts != len(years)).sum()
print(f"{n_incomplete} pixels missing at least one year after cleaning")

Clean table: 53308068 rows, 2423094 pixels
0 pixels missing at least one year after cleaning


In [7]:
# ======================================================================
# ---------- Step 5: export tables ----------
# ======================================================================

drop_helper_cols = ["lon_r", "lat_r"]

# yearly tables
for y in years:
    yearly = clean_df.loc[clean_df["year"] == y].drop(columns=drop_helper_cols)
    yearly.to_parquet(out_dir_tables / f"table_wgs84_{y}_clean.parquet", index=False)
print(f"Saved {len(years)} yearly cleaned tables to {out_dir_tables}")

# one big combined table
big_path = out_dir_tables / "table_wgs84_2002-2023_clean.parquet"
clean_df.drop(columns=drop_helper_cols).to_parquet(big_path, index=False)
print(f"Saved combined cleaned table ({len(clean_df)} rows) to {big_path}")

# small example table: a handful of pixels, all their years
example_ids = good_pixels[ID_COL].sample(
    n=min(N_EXAMPLE_PIXELS, len(good_pixels)), random_state=RANDOM_SEED
)
example_df = clean_df.loc[clean_df[ID_COL].isin(example_ids)].drop(columns=drop_helper_cols)
example_path = out_dir_tables / "table_wgs84_example.parquet"
example_df.to_parquet(example_path, index=False)
print(f"Saved example table ({len(example_df)} rows, {len(example_ids)} pixels) to {example_path}")

Saved 22 yearly cleaned tables to /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new
Saved combined cleaned table (53308068 rows) to /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/table_wgs84_2002-2023_clean.parquet
Saved example table (2200000 rows, 100000 pixels) to /Users/Wanja/Documents/non-equilibrium_data/tables_wgs84_new/table_wgs84_example.parquet


In [ ]:
# ======================================================================
# ---------- Step 6: clean rasters using the SAME pixel ID ----------
# ======================================================================


def build_id_grid(transform, width, height, pixel_table, id_col=ID_COL, lon_col="lon_r", lat_col="lat_r"):
    """Rasterize the (lon_r, lat_r) -> location_id lookup onto a grid defined by `transform`."""
    rows, cols = rasterio.transform.rowcol(
        transform, pixel_table[lon_col].values, pixel_table[lat_col].values
    )
    rows = np.asarray(rows)
    cols = np.asarray(cols)

    valid = (rows >= 0) & (rows < height) & (cols >= 0) & (cols < width)
    if not valid.all():
        print(f"WARNING: {(~valid).sum()} lookup pixels fall outside this raster's grid")

    id_grid = np.full((height, width), -1, dtype=np.int64)
    id_grid[rows[valid], cols[valid]] = pixel_table.loc[valid, id_col].values
    return id_grid


id_grid = None  # built once from the first year, reused for every year (grid is assumed identical across years)

for y in years:
    with rasterio.open(tif_paths[y]) as src:
        profile = src.profile.copy()
        descriptions = src.descriptions
        data = src.read(masked=True)  # (bands, height, width)
        transform = src.transform
        height, width = src.height, src.width
        src_nodata = src.nodata

    if id_grid is None:
        id_grid = build_id_grid(transform, width, height, good_pixels)

        # write the static ID raster once (identical for every year)
        id_profile = profile.copy()
        id_profile.update(count=1, dtype="int32", nodata=-1)
        with rasterio.open(out_dir_rasters / "location_id.tif", "w", **id_profile) as dst:
            dst.write(id_grid.astype("int32")[np.newaxis, :, :])
            dst.set_band_description(1, "location_id")
        print(f"Saved static location ID raster to {out_dir_rasters / 'location_id.tif'}")

    keep_mask = id_grid != -1  # True = kept pixel

    cleaned = data.copy()
    cleaned[:, ~keep_mask] = np.ma.masked
    nodata_value = src_nodata if src_nodata is not None else np.nan
    cleaned_filled = cleaned.filled(nodata_value)

    out_profile = profile.copy()
    out_profile["nodata"] = nodata_value
    out_path = out_dir_rasters / f"stack_wgs84_{y}_clean.tif"
    with rasterio.open(out_path, "w", **out_profile) as dst:
        dst.write(cleaned_filled)
        for i, name in enumerate(descriptions, start=1):
            dst.set_band_description(i, name)

    print(f"Year {y}: wrote cleaned raster ({keep_mask.sum()} kept pixels) to {out_path}")

print("Done.")

Saved static location ID raster to /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/location_id.tif
Year 2002: wrote cleaned raster (2423094 kept pixels) to /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_2002_clean.tif
Year 2003: wrote cleaned raster (2423094 kept pixels) to /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_2003_clean.tif
Year 2004: wrote cleaned raster (2423094 kept pixels) to /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_2004_clean.tif
Year 2005: wrote cleaned raster (2423094 kept pixels) to /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_2005_clean.tif
Year 2006: wrote cleaned raster (2423094 kept pixels) to /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_2006_clean.tif
Year 2007: wrote cleaned raster (2423094 kept pixels) to /Users/Wanja/Documents/non-equilibrium_data/rasters_wgs84_new/stack_wgs84_2007_clean.tif
Year 2008: 

: 